# THEMAP — API deep dive

<a href="https://colab.research.google.com/github/HFooladi/THEMAP/blob/main/notebooks/colab/02_api_deep_dive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[`01_quick_tour.ipynb`](01_quick_tour.ipynb) showed the one-call `quick_distance` shortcut. This notebook opens the hood and walks through the building blocks that shortcut composes:

| Layer | Class | What it does |
|---|---|---|
| Data   | `DatasetLoader`, `MoleculeDataset`     | discover and load `*.jsonl.gz` files |
| Features | `MoleculeFeaturizer`                 | turn SMILES into numpy feature matrices |
| Distance | `DatasetDistance`                    | compute an N×M dataset-distance matrix |
| Pipeline | `Pipeline` + YAML config             | wire it all together for reproducible runs |

You will also compare two featurizer/metric combinations side by side and produce a 2D map of all tasks via PCA on their dataset prototypes.

Runtime: ~5 minutes on CPU.

## 1. Install and fetch sample data

Same setup as the quick tour, plus `scikit-learn` (already a `themap` transitive dep, just being explicit) for the PCA at the end. We pull 10 source + 3 target ChEMBL datasets — enough to make the landscape plot interesting.

In [ ]:
%pip install -q themap molfeat rdkit matplotlib seaborn scikit-learn

In [ ]:
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/HFooladi/THEMAP/main/datasets"
SAMPLES = {
    "train": [
        "CHEMBL1023359",
        "CHEMBL1613776",
        "CHEMBL1614274",
        "CHEMBL2218944",
        "CHEMBL2219012",
        "CHEMBL3371729",
        "CHEMBL3705844",
        "CHEMBL3866221",
        "CHEMBL4224224",
        "CHEMBL894522",
    ],
    "test": ["CHEMBL1963831", "CHEMBL2219236", "CHEMBL2219358"],
}

for fold, ids in SAMPLES.items():
    os.makedirs(f"datasets/{fold}", exist_ok=True)
    for cid in ids:
        dst = f"datasets/{fold}/{cid}.jsonl.gz"
        if not os.path.exists(dst):
            urllib.request.urlretrieve(f"{BASE}/{fold}/{cid}.jsonl.gz", dst)

print("train tasks:", sorted(os.listdir("datasets/train")))
print("test tasks: ", sorted(os.listdir("datasets/test")))

## 2. Load all tasks with `DatasetLoader`

`DatasetLoader` is the front door for batch loading. Point it at the root and call `load_datasets(fold)` for each fold; you get back a `dict[task_id -> MoleculeDataset]`.

In [ ]:
from themap import DatasetLoader

loader = DatasetLoader("datasets")
train_ds = loader.load_datasets("train")
test_ds = loader.load_datasets("test")

print(f"{len(train_ds)} train + {len(test_ds)} test tasks loaded\n")
print(f"{'task_id':<20} {'#mols':>6}  {'pos%':>6}")
for tid, d in sorted({**train_ds, **test_ds}.items()):
    print(f"{tid:<20} {len(d):>6}  {d.positive_ratio:>6.1%}")

## 3. Featurize with `MoleculeFeaturizer`

`MoleculeFeaturizer` wraps `molfeat`. It deduplicates SMILES across datasets before calling the underlying transformer, so featurizing 13 tasks is faster than featurizing each in isolation.

Call `featurize_datasets({task_id: dataset})` once and store the feature arrays on each dataset via `set_features(...)`. After this step the datasets are ready for distance computation.

In [ ]:
from themap import MoleculeFeaturizer

fp = MoleculeFeaturizer("ecfp", n_jobs=2)

all_ds = {**train_ds, **test_ds}
features = fp.featurize_datasets(all_ds, deduplicate=True)
for tid, X in features.items():
    all_ds[tid].set_features(X, "ecfp")

example_tid = next(iter(features))
print(
    f"feature matrix for {example_tid}: shape={features[example_tid].shape}, dtype={features[example_tid].dtype}"
)

## 4. Compute distances with `DatasetDistance`

This is the lowest-level entry point. You hand it parallel lists of feature arrays, label arrays, and ids; it returns the nested dict `target_id -> source_id -> distance`. The pipeline uses this internally, but going direct is useful when features come from somewhere outside THEMAP (your own embeddings, a fine-tuned model, etc.).

In [ ]:
import pandas as pd

from themap import DatasetDistance


def matrix_df(feature_dict, train_ds, test_ds, method):
    src_ids = sorted(train_ds)
    tgt_ids = sorted(test_ds)
    calc = DatasetDistance(method=method)
    nested = calc.compute_matrix(
        source_features=[feature_dict[i] for i in src_ids],
        source_labels=[train_ds[i].labels for i in src_ids],
        target_features=[feature_dict[i] for i in tgt_ids],
        target_labels=[test_ds[i].labels for i in tgt_ids],
        source_ids=src_ids,
        target_ids=tgt_ids,
    )
    df = pd.DataFrame(nested).T  # rows=targets, cols=sources
    df.index.name, df.columns.name = "target", "source"
    return df


ecfp_euclid = matrix_df(features, train_ds, test_ds, "euclidean")
ecfp_euclid.round(3)

## 5. Compare featurizers and metrics

Different fingerprints encode molecules differently, and different metrics emphasise different things. Below we recompute the matrix with **MACCS keys** (a 167-bit knowledge-based fingerprint) and **cosine** distance, then sit the two heatmaps side by side to see whether the *ranking* of nearest sources is stable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

maccs_fp = MoleculeFeaturizer("maccs", n_jobs=2).featurize_datasets(all_ds)
maccs_cos = matrix_df(maccs_fp, train_ds, test_ds, "cosine")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.heatmap(ecfp_euclid, annot=True, fmt=".2f", cmap="viridis_r", ax=axes[0])
axes[0].set_title("ECFP + Euclidean")
sns.heatmap(maccs_cos, annot=True, fmt=".2f", cmap="viridis_r", ax=axes[1])
axes[1].set_title("MACCS + Cosine")
plt.tight_layout()
plt.show()

print("\nNearest source per target (different featurizers may disagree):")
print(f"{'target':<16} {'ECFP/Euclid':<18} {'MACCS/Cos':<18}")
for tgt in ecfp_euclid.index:
    a = ecfp_euclid.loc[tgt].idxmin()
    b = maccs_cos.loc[tgt].idxmin()
    mark = "" if a == b else "  <- disagreement"
    print(f"{tgt:<16} {a:<18} {b:<18}{mark}")

## 6. Wire it together with a YAML config

For reproducible experiments — and to share runs with colleagues — describe the whole pipeline in YAML and execute it with one call. The YAML is the single source of truth, version-controllable alongside your code.

In [ ]:
from pathlib import Path

from themap import run_pipeline

Path("my_config.yaml").write_text("""\
data:
  directory: datasets
molecule:
  enabled: true
  featurizer: ecfp
  method: euclidean
output:
  directory: output_yaml
  format: csv
compute:
  n_jobs: 2
  device: auto
""")

results = run_pipeline("my_config.yaml")
print("saved files:", sorted(os.listdir("output_yaml")))
print("matrix keys:", list(results.keys()))

## 7. Plot the task landscape

Each task's "signature" — what `DatasetDistance` actually compares — is the concatenation of its positive and negative class **prototypes** (mean feature vectors per class). Reducing those signatures to 2D with PCA gives a quick map of which tasks live near each other in feature space.

With 13 tasks PCA is the right call (UMAP needs more points to be reliable). Color encodes the fold (train vs test).

In [ ]:
import numpy as np
from sklearn.decomposition import PCA


def prototype(X, y):
    pos = X[y == 1].mean(axis=0) if (y == 1).any() else np.zeros(X.shape[1])
    neg = X[y == 0].mean(axis=0) if (y == 0).any() else np.zeros(X.shape[1])
    return np.concatenate([pos, neg])


ids = sorted(all_ds)
sigs = np.stack([prototype(features[t], all_ds[t].labels) for t in ids])
coords = PCA(n_components=2).fit_transform(sigs)

fig, ax = plt.subplots(figsize=(6, 5))
for fold, marker, color in [("train", "o", "#1f77b4"), ("test", "^", "#d62728")]:
    mask = [t in (train_ds if fold == "train" else test_ds) for t in ids]
    ax.scatter(coords[mask, 0], coords[mask, 1], marker=marker, color=color, s=80, label=fold, edgecolor="k")
for (x, y), t in zip(coords, ids):
    ax.annotate(t.replace("CHEMBL", ""), (x, y), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("Task landscape (ECFP prototypes)")
ax.legend()
plt.tight_layout()
plt.show()

## Where to go next

What you saw here scales up in several directions:

- **More featurizers** — `themap list-featurizers` enumerates 27 molecule featurizers: fingerprints (ECFP, MACCS, Avalon, …), count fingerprints, RDKit descriptors (`desc2D`, `mordred`), and neural embeddings (ChemBERTa, MolT5, GIN). Neural ones generally want a GPU.
- **OTDD** — for label-aware dataset distances, pass `method="otdd"`. OTDD pulls in `pot` + `geomloss` and needs a GPU at this scale; it is also sensitive to the featurizer (use continuous descriptors or neural embeddings, not raw binary fingerprints). See the paper for details.
- **Protein distances** — if your `datasets/` folder also contains `<task>.fasta` files, add a `protein:` block to the YAML to compute ESM-2-based target-side distances and combine them with the molecule distances.
- **CLI** — every block above has a `themap` CLI counterpart: `themap featurize`, `themap quick`, `themap run my_config.yaml`. See `themap --help` or the [CLI section of the README](https://github.com/HFooladi/THEMAP#cli-reference).
- **Full FS-Mol reproduction** — the paper-companion notebooks in `notebooks/` and the [Reproducing FS-Mol Experiments](https://github.com/HFooladi/THEMAP#reproducing-fs-mol-experiments) section of the README walk through the full benchmark.
- **Docs** — [hfooladi.github.io/THEMAP](https://hfooladi.github.io/THEMAP/).